<div style="font-size:2em; font-weight:bold; margin-bottom:8px;">01 — Download & Normalize the SciFact Corpus</div>

This notebook downloads the **BEIR SciFact corpus** from Hugging Face and saves it as a clean, normalized JSONL file.

It is **Step 1** of the RAG data indexing pipeline — raw data acquisition only.

---

**What this notebook does:**
1. Loads the corpus from Hugging Face using the `datasets` library
2. Explores the raw data so you can see what you are working with
3. Checks data quality before processing
4. Defines simple helper functions for validation and normalization
5. Saves the full corpus to `data/raw/corpus/corpus.jsonl`
6. Reads the output file back to verify it looks correct

**What this notebook intentionally does NOT do:**
- No chunking
- No embedding
- No Qdrant indexing

> **Before running:** install the dependencies first.
> ```bash
> pip install -r requirements.txt
> ```

---
## 1. Imports

We only need three things:
- **`json`** — to serialize each document as a JSON string when writing the output file
- **`pathlib.Path`** — to handle file and directory paths in a clean, OS-independent way
- **`load_dataset`** — the Hugging Face function that downloads and caches datasets

In [3]:
# ── [1 / 10] Imports ────────────────────────────────────────────────────────

import json
from pathlib import Path

from datasets import load_dataset

---
## 2. Configuration — Dataset and Paths

All configurable values live here in one place.
If you want to change the dataset or output location later, edit only this cell.

**Path logic:**
The notebook may run with its working directory set to `notebooks/` (when launched from the terminal)
or to the project root (when run inside VS Code / Cursor).
The two-line path detection below handles both cases automatically.

In [4]:
# ── [2 / 10] Configuration ──────────────────────────────────────────────────

# Dataset settings
DATASET_NAME   = "BeIR/scifact"
DATASET_CONFIG = "corpus"

# Resolve the project root no matter where the notebook is run from
_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

# Output paths
OUTPUT_DIR  = ROOT / "data" / "raw" / "corpus"
OUTPUT_FILE = OUTPUT_DIR / "corpus.jsonl"

print(f"Project root : {ROOT}")
print(f"Output file  : {OUTPUT_FILE}")

Project root : /app
Output file  : /app/data/raw/corpus/corpus.jsonl


---
## 3. Load the Dataset from Hugging Face

`load_dataset` will download the corpus on the first run and store it in a local cache (`~/.cache/huggingface/`).
The next time you run this cell, it loads instantly from the cache — no re-download.

We load only the **`corpus`** split because this repository focuses on document indexing.
Queries and qrels belong to the retrieval benchmark repository.

In [5]:
# ── [3 / 10] Load the Dataset from Hugging Face ─────────────────────────────

print(f"Loading '{DATASET_NAME}' (config: '{DATASET_CONFIG}') from Hugging Face...")
print()

dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)

print()
print("Dataset loaded successfully:")
print(dataset)

Loading 'BeIR/scifact' (config: 'corpus') from Hugging Face...



Generating corpus split: 100%|██████████| 5183/5183 [00:00<00:00, 82338.44 examples/s]


Dataset loaded successfully:
DatasetDict({
    corpus: Dataset({
        features: ['_id', 'title', 'text'],
        num_rows: 5183
    })
})


---
## 4. Explore the Raw Data

Before processing anything, let's look at what we downloaded.

We want to know:
- Which split keys exist inside the dataset object
- How many documents are in the corpus
- What fields (columns) each document has
- What real documents look like

In [6]:
# ── [4 / 10] Explore — Dataset Info ─────────────────────────────────────────

# Get the corpus split — the key name can vary across dataset versions
split_name = "corpus" if "corpus" in dataset else list(dataset.keys())[0]
corpus = dataset[split_name]

print(f"Split used : {split_name!r}")
print(f"Total docs : {len(corpus):,}")
print(f"Fields     : {corpus.column_names}")

Split used : 'corpus'
Total docs : 5,183
Fields     : ['_id', 'title', 'text']


In [7]:
# ── [5 / 10] Explore — Preview First 3 Documents ────────────────────────────

# Preview the first 3 documents to understand the raw structure
print("=" * 60)
print("First 3 raw documents")
print("=" * 60)

for i, row in enumerate(corpus.select(range(3))):
    print(f"\n--- Document {i} ---")
    print(f"  _id   : {row['_id']}")
    print(f"  title : {row['title']}")
    # Limit text preview to 200 characters so the output stays readable
    text_preview = row["text"][:200].replace("\n", " ")
    print(f"  text  : {text_preview}...")

First 3 raw documents

--- Document 0 ---
  _id   : 4983
  title : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
  text  : Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic re...

--- Document 1 ---
  _id   : 5836
  title : Induction of myelodysplasia by myeloid-derived suppressor cells.
  text  : Myelodysplastic syndromes (MDS) are age-dependent stem cell malignancies that share biological features of activated adaptive immune response and ineffective hematopoiesis. Here we report that myeloid...

--- Document 2 ---
  _id   : 7912
  title : BC1 RNA, the transcript from a master gene for ID element amplification, is able to prime its own reverse transcription.
  text  : ID elements are short interspersed elements (SINEs) found in high copy number in many r

---
## 5. Data Quality Check

Before saving, let's scan all rows for two kinds of problems:

| Problem | How we handle it |
|---|---|
| Missing `_id` | Raise an error — a document without an ID breaks the pipeline |
| Empty `title` AND `text` | Skip — nothing useful to index |

This step also gives you confidence in the data quality before you process 5,000+ rows.

In [8]:
# ── [6 / 10] Data Quality Check ─────────────────────────────────────────────

missing_id    = sum(1 for row in corpus if not row.get("_id"))
empty_content = sum(
    1 for row in corpus
    if not (row.get("title") or "").strip()
    and not (row.get("text")  or "").strip()
)

print(f"Rows missing '_id'         : {missing_id}")
print(f"Rows with empty title+text : {empty_content}")
print()

if missing_id == 0 and empty_content == 0:
    print("All documents look clean — no issues found.")

Rows missing '_id'         : 0
Rows with empty title+text : 0

All documents look clean — no issues found.


---
## 6. Helper Functions — Validation and Normalization

We define three small, focused functions. Each one does exactly one thing.

| Function | Responsibility |
|---|---|
| `validate_row` | Raise an error if `_id` is missing |
| `should_skip` | Return `True` if both `title` and `text` are empty |
| `normalize_row` | Map Hugging Face fields to our project's document schema |

Keeping these as separate functions makes them easy to test individually
and easy to reuse in future notebooks.

In [9]:
# ── [7 / 10] Helper Functions ────────────────────────────────────────────────

def validate_row(row: dict) -> None:
    """Raise ValueError if the row has no document ID."""
    if not row.get("_id"):
        raise ValueError(f"Row is missing '_id': {row}")


def should_skip(row: dict) -> bool:
    """Return True if the row has no usable content at all."""
    title = (row.get("title") or "").strip()
    text  = (row.get("text")  or "").strip()
    return not title and not text


def normalize_row(row: dict) -> dict:
    """Convert a raw Hugging Face row to the project's normalized document schema."""
    return {
        "document_id"    : row["_id"],
        "title"          : row.get("title", ""),
        "text"           : row.get("text", ""),
        "source"         : DATASET_NAME,
        "dataset_config" : DATASET_CONFIG,
    }


print("Functions defined.")

Functions defined.


---
## 7. Test the Functions on a Single Document

Always test your logic on one example before running it on thousands of rows.

Here we take the first document from the corpus and run all three functions on it
so we can see exactly what the output looks like before writing the file.

In [10]:
# ── [8 / 10] Test Functions on a Single Document ────────────────────────────

sample = corpus[0]

print("Raw input row:")
print(sample)
print()

# Test validate_row — should pass silently
validate_row(sample)
print("validate_row  : passed")

# Test should_skip — should be False for a real document
print(f"should_skip   : {should_skip(sample)}  (expected: False)")

# Test normalize_row — show the resulting document
normalized = normalize_row(sample)
print()
print("Normalized output:")
print(json.dumps(normalized, indent=2, ensure_ascii=False))

Raw input row:
{'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the intern

---
## 8. Save the Full Corpus to JSONL

Now we process all documents and write them to `data/raw/corpus/corpus.jsonl`.

**Why JSONL?**
JSONL (JSON Lines) stores one JSON object per line. This format is ideal for pipelines because:
- You can stream it line by line without loading the whole file into memory
- It is easy to append to
- Any tool can read it — Python, pandas, jq, etc.

This file is the input to the next pipeline step: text cleaning and chunking.

In [11]:
# ── [9 / 10] Save the Full Corpus to JSONL ──────────────────────────────────

# Create the output folder if it does not already exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

saved   = 0
skipped = 0

with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for row in corpus:

        # Raise immediately if the document ID is missing
        validate_row(row)

        # Skip rows that have absolutely no content
        if should_skip(row):
            skipped += 1
            continue

        doc  = normalize_row(row)
        # ensure_ascii=False preserves non-ASCII characters (e.g. Greek letters in paper titles)
        line = json.dumps(doc, ensure_ascii=False)
        f.write(line + "\n")
        saved += 1

print(f"Saved   : {saved:,} documents")
if skipped:
    print(f"Skipped : {skipped} rows (empty title and text)")
print(f"Output  : {OUTPUT_FILE.resolve()}")

Saved   : 5,183 documents
Output  : /app/data/raw/corpus/corpus.jsonl


---
## 9. Verify the Output File

Let's read the file back and confirm:
- It contains the expected number of lines
- Each line is valid JSON
- The document schema matches what we defined

In [12]:
# ── [10 / 10] Verify the Output File ────────────────────────────────────────

with OUTPUT_FILE.open("r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Lines in output file : {len(lines):,}")
print()
print("=" * 60)
print("First 3 documents from corpus.jsonl")
print("=" * 60)

for i, line in enumerate(lines[:3]):
    doc = json.loads(line)
    print(f"\n--- Document {i} ---")
    for key, value in doc.items():
        # Trim long values so the output stays readable
        display = str(value)[:100] + "..." if len(str(value)) > 100 else value
        print(f"  {key:<16} : {display}")

Lines in output file : 5,183

First 3 documents from corpus.jsonl

--- Document 0 ---
  document_id      : 4983
  title            : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion ten...
  text             : Alterations of the architecture of cerebral white matter in the developing human brain can affect co...
  source           : BeIR/scifact
  dataset_config   : corpus

--- Document 1 ---
  document_id      : 5836
  title            : Induction of myelodysplasia by myeloid-derived suppressor cells.
  text             : Myelodysplastic syndromes (MDS) are age-dependent stem cell malignancies that share biological featu...
  source           : BeIR/scifact
  dataset_config   : corpus

--- Document 2 ---
  document_id      : 7912
  title            : BC1 RNA, the transcript from a master gene for ID element amplification, is able to prime its own re...
  text             : ID elements are short interspersed elements (SINEs) found in high